<a href="https://colab.research.google.com/github/subashravircse2024-beep/CIT/blob/main/ex_no_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================
# RAG Implementation in Google Colab
# Sentence Transformers + FAISS + FLAN-T5
# =====================================================

# -----------------------------
# 1. Install Required Packages
# -----------------------------
!pip install -q sentence-transformers faiss-cpu transformers sentencepiece accelerate


# -----------------------------
# 2. Import Libraries
# -----------------------------
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# -----------------------------
# 3. Knowledge Base
# -----------------------------
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",

    "Retrieval-Augmented Generation (RAG) combines document retrieval with text generation. "
    "It retrieves relevant information from documents and uses a language model to generate answers.",

    "Python is a popular high-level programming language used in artificial intelligence, "
    "machine learning, and data science.",

    "Vector databases store embeddings and allow fast similarity search between documents and queries."
]


print("Documents Loaded:", len(documents))


# -----------------------------
# 4. Load Embedding Model
# -----------------------------
print("\nLoading embedding model...")

embed_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# -----------------------------
# 5. Create Document Embeddings
# -----------------------------
doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

doc_embeddings = np.array(
    doc_embeddings,
    dtype="float32"
)


print("Embedding Shape:", doc_embeddings.shape)


# -----------------------------
# 6. Create FAISS Vector Index
# -----------------------------
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(doc_embeddings)


print("FAISS Index Created")
print("Total Vectors:", index.ntotal)



# -----------------------------
# 7. User Question
# -----------------------------
query = "What is RAG in AI?"


query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = np.array(
    query_embedding,
    dtype="float32"
)



# -----------------------------
# 8. Retrieve Relevant Documents
# -----------------------------
top_k = 2

scores, ids = index.search(
    query_embedding,
    top_k
)


retrieved_documents = [
    documents[i] for i in ids[0]
]


print("\nRetrieved Documents:")
for doc in retrieved_documents:
    print("-", doc)



# -----------------------------
# 9. Create RAG Prompt
# -----------------------------
context = "\n".join(retrieved_documents)


prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""


# -----------------------------
# 10. Load FLAN-T5 Model
# -----------------------------
print("\nLoading FLAN-T5 model...")

model_name = "google/flan-t5-base"


tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)



# -----------------------------
# 11. Generate Answer
# -----------------------------
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)


outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)


answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)



# -----------------------------
# 12. Final Output
# -----------------------------
print("\n==============================")
print("QUESTION")
print("==============================")
print(query)


print("\n==============================")
print("ANSWER")
print("==============================")
print(answer)


Documents Loaded: 4

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Shape: (4, 384)
FAISS Index Created
Total Vectors: 4

Retrieved Documents:
- Retrieval-Augmented Generation (RAG) combines document retrieval with text generation. It retrieves relevant information from documents and uses a language model to generate answers.
- Python is a popular high-level programming language used in artificial intelligence, machine learning, and data science.

Loading FLAN-T5 model...


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


QUESTION
What is RAG in AI?

ANSWER
combines document retrieval with text generation
